### Ausreißerkontrolle

In [1]:
import pandas as pd
import numpy as np
from openpyxl import load_workbook
from openpyxl.styles import PatternFill
import os

def mark_outliers(input_path, z_limit=4, iqr_factor=2.5):
    """
    Markiert Outlier in numerischen Spalten einer Excel-Datei.
    - z_limit: Schwelle für |z| (Standard: 4)
    - iqr_factor: Multiplikator für IQR (Standard: 2.0)
    """

    # === 1. Datei einlesen ===
    df = pd.read_excel(input_path)

    # Nur numerische Spalten betrachten
    num_cols = df.select_dtypes(include=[np.number]).columns

    # === 2. Outlier-Detection vorbereiten ===
    zscore_outliers = pd.DataFrame(False, index=df.index, columns=num_cols)
    iqr_outliers = pd.DataFrame(False, index=df.index, columns=num_cols)

    # Z-Score Outliers
    for col in num_cols:
        mean, std = df[col].mean(), df[col].std()
        if std > 0:  # Division by zero vermeiden
            zscore_outliers[col] = np.abs((df[col] - mean) / std) > z_limit

    # IQR Outliers
    for col in num_cols:
        q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
        iqr = q3 - q1
        lower, upper = q1 - iqr_factor * iqr, q3 + iqr_factor * iqr
        iqr_outliers[col] = (df[col] < lower) | (df[col] > upper)

    # === 3. Output-Datei bestimmen ===
    base, ext = os.path.splitext(input_path)
    output_path = f"{base}_outlier{ext}"

    # Kopie speichern
    df.to_excel(output_path, index=False)

    # === 4. Mit openpyxl farblich markieren ===
    wb = load_workbook(output_path)
    ws = wb.active

    # Farben definieren
    yellow_fill  = PatternFill(start_color="FFFF00", end_color="FFFF00", fill_type="solid")  # nur Z-Score
    orange_fill  = PatternFill(start_color="FFA500", end_color="FFA500", fill_type="solid")  # nur IQR
    red_fill     = PatternFill(start_color="FF0000", end_color="FF0000", fill_type="solid")  # beide

    any_outliers = False
    outlier_summary = {col: 0 for col in num_cols}

    for j, col in enumerate(num_cols, start=1):
        for i in range(len(df)):
            row_excel = i + 2  # +1 für Header, +1 für 0-based Index
            col_excel = j      # Korrektur: nicht j+1!
            zflag = zscore_outliers.iloc[i, j-1]
            iflag = iqr_outliers.iloc[i, j-1]

            if zflag and iflag:
                ws.cell(row=row_excel, column=col_excel).fill = red_fill
                any_outliers = True
                outlier_summary[col] += 1
            elif zflag:
                ws.cell(row=row_excel, column=col_excel).fill = yellow_fill
                any_outliers = True
                outlier_summary[col] += 1
            elif iflag:
                ws.cell(row=row_excel, column=col_excel).fill = orange_fill
                any_outliers = True
                outlier_summary[col] += 1


    wb.save(output_path)

    # === 5. Print-Ausgabe ===
    if any_outliers:
        print(f"⚠️  Outlier(s) gefunden! Ergebnis gespeichert unter: {output_path}")
        print("Zusammenfassung (Spalte: Anzahl Ausreißer):")
        for col, n in outlier_summary.items():
            if n > 0:
                print(f"  - {col}: {n}")
        print("\nLegende der Farben:")
        print("  🟨 Gelb   = Z-Score Outlier (|z| > {z_limit})")
        print("  🟧 Orange = IQR Outlier (außerhalb Q1 ± {iqr_factor}·IQR)")
        print("  🟥 Rot    = Outlier nach beiden Methoden")
    else:
        print(f"✅ Keine Outlier gefunden. Datei gespeichert unter: {output_path}")


def main():
    # 👉 Hier deinen Pfad einsetzen
    input_file = r"K:\Team\Böhmer_Michael\PAPER\Philipp\Datensatz\Basistabelle_5_bis_14_PostOP.xlsx"

    # Schwellenwerte kannst du hier anpassen
    mark_outliers(input_file, z_limit=4, iqr_factor=2.5)


if __name__ == "__main__":
    main()

⚠️  Outlier(s) gefunden! Ergebnis gespeichert unter: K:\Team\Böhmer_Michael\PAPER\Philipp\Datensatz\Basistabelle_5_bis_14_PostOP_outlier.xlsx
Zusammenfassung (Spalte: Anzahl Ausreißer):
  - CMJ_Vertical Takeoff velocity: 1
  - CMJ_Jump Height impulse: 1
  - CMJ_maxRFD: 2
  - CMJ_Vertical Stiffness: 5
  - CMJ_Propulsive duration: 1
  - CMJ_Rel. Peak landing force: 3
  - INV_CMJ_uni_Rel. Peak landing force : 5
  - UNINV_CMJ_uni_Rel. Peak landing force : 2
  - CMJ_TTS: 2
  - INV_CMJ_uni_Peak Loading Force-Mittelwert [N]: 1
  - INV_CMJ_uni_Vertical Take-Off Velocity by Net Impulse-Mittelwert [m/s]: 1
  - INV_CMJ_uni_Vertical Stiffness-Mittelwert [kN/m]: 6
  - INV_CMJ_uni_Braking Duration-Mittelwert [s]: 1
  - INV_CMJ_uni_Countermovement Time-Mittelwert [s]: 2
  - INV_CMJ_uni_Peak Braking Force-Mittelwert [N]: 1
  - INV_CMJ_uni_Durchschnittliche Bremskraft-Mittelwert [N]: 1
  - INV_CMJ_uni_Peak Propulsive Force-Mittelwert [N]: 3
  - INV_CMJ_uni_Durchschnittliche Schubkraft-Mittelwert [N]: 3

### Modellauswahl

In [1]:
# # ============================ Robust CV + Bootstrap-Table ============================
# import warnings
# warnings.filterwarnings("ignore")

# import numpy as np
# import pandas as pd
# from IPython.display import HTML, display

# from sklearn.model_selection import RepeatedStratifiedKFold
# from sklearn.pipeline import Pipeline
# from sklearn.preprocessing import StandardScaler, LabelEncoder
# from sklearn.impute import SimpleImputer
# from sklearn.metrics import (
#     accuracy_score, f1_score, roc_auc_score, recall_score, precision_score,
#     balanced_accuracy_score, average_precision_score, brier_score_loss, log_loss
# )

# from sklearn.linear_model import LogisticRegression
# from sklearn.tree import DecisionTreeClassifier
# from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier, BaggingClassifier
# from xgboost import XGBClassifier
# from lightgbm import LGBMClassifier
# from sklearn.svm import SVC
# from sklearn.neighbors import KNeighborsClassifier
# from sklearn.neural_network import MLPClassifier
# from sklearn.naive_bayes import GaussianNB
# from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis

# # ---------------- Utilities ----------------
# def proba_or_decision(model, X):
#     """Bevorzuge predict_proba; sonst decision_function (auf [0,1] gemappt); sonst Pseudo-Score."""
#     if hasattr(model, "predict_proba"):
#         return model.predict_proba(X)[:, 1]
#     if hasattr(model, "decision_function"):
#         s = model.decision_function(X)
#         s_min, s_max = np.min(s), np.max(s)
#         return (s - s_min) / (s_max - s_min + 1e-12)
#     # Fallback: harte Labels -> {0,1}
#     return (model.predict(X) == 1).astype(float)

# def repeated_k_fold_robust(base_estimator, X_df: pd.DataFrame, y_vec, n_splits=5, n_repeats=10):
#     # Binäres Label robust kodieren
#     le = LabelEncoder()
#     y = le.fit_transform(y_vec)
#     if len(le.classes_) != 2:
#         raise ValueError("Dieses Setup erwartet binäre Klassifikation (2 Klassen).")

#     rkf = RepeatedStratifiedKFold(n_splits=n_splits, n_repeats=n_repeats, random_state=42)

#     # Pipeline: Imputer + Scaler + Modell
#     pipe = Pipeline(steps=[
#         ("impute", SimpleImputer(strategy="median")),
#         ("scale",  StandardScaler(with_mean=True, with_std=True)),
#         ("clf",    base_estimator)
#     ])

#     m = {
#         "acc_tr":[], "acc_te":[], "bacc":[],
#         "f1":[], "rec":[], "prec":[],
#         "roc":[], "prauc":[],
#         "brier":[], "logloss":[]
#     }

#     X = X_df.values

#     for tr, te in rkf.split(X, y):
#         Xtr, Xte = X[tr], X[te]
#         ytr, yte = y[tr], y[te]

#         pipe.fit(Xtr, ytr)

#         ytr_pred = pipe.predict(Xtr)
#         yte_pred = pipe.predict(Xte)
#         yte_score = proba_or_decision(pipe, Xte)
#         yte_score = np.clip(yte_score, 1e-7, 1-1e-7)  # stabil für Brier/LogLoss

#         m["acc_tr"].append(accuracy_score(ytr, ytr_pred))
#         m["acc_te"].append(accuracy_score(yte, yte_pred))
#         m["bacc"].append(balanced_accuracy_score(yte, yte_pred))

#         m["f1"].append(f1_score(yte, yte_pred))
#         m["rec"].append(recall_score(yte, yte_pred))
#         m["prec"].append(precision_score(yte, yte_pred))

#         m["roc"].append(roc_auc_score(yte, yte_score))
#         m["prauc"].append(average_precision_score(yte, yte_score))

#         m["brier"].append(brier_score_loss(yte, yte_score))
#         m["logloss"].append(log_loss(yte, yte_score))

#     # Means & STDs
#     out = {k: (np.mean(v), np.std(v)) for k, v in m.items()}
#     return out


# def build_results_table(results_dict):
#     """
#     Baut zwei DataFrames:
#       - df_view: formatiert (mean ± std) in gewünschter Reihenfolge
#       - df_num:  numerisch (means/stds), nach ROC-AUC_mean sortiert
#     """
#     rows = []
#     for name, m in results_dict.items():
#         rows.append({
#             "Model": name,
#             "TrainAcc_mean":  m["acc_tr"][0], "TrainAcc_std":  m["acc_tr"][1],
#             "TestAcc_mean":   m["acc_te"][0], "TestAcc_std":   m["acc_te"][1],
#             "BalAcc_mean":    m["bacc"][0],   "BalAcc_std":    m["bacc"][1],
#             "F1_mean":        m["f1"][0],     "F1_std":        m["f1"][1],
#             "Recall_mean":    m["rec"][0],    "Recall_std":    m["rec"][1],
#             "Precision_mean": m["prec"][0],   "Precision_std": m["prec"][1],
#             "ROC_mean":       m["roc"][0],    "ROC_std":       m["roc"][1],
#             "PRAUC_mean":     m["prauc"][0],  "PRAUC_std":     m["prauc"][1],
#             "Brier_mean":     m["brier"][0],  "Brier_std":     m["brier"][1],
#             "LogLoss_mean":   m["logloss"][0],"LogLoss_std":   m["logloss"][1],
#         })
#     df_num = pd.DataFrame(rows).sort_values("ROC_mean", ascending=False).reset_index(drop=True)

#     def fmt(mean, std): 
#         return f"{mean:.3f} ± {std:.3f}"

#     df_view = pd.DataFrame({
#         "Model":           df_num["Model"],
#         "Train Accuracy":  [fmt(m, s) for m, s in zip(df_num["TrainAcc_mean"],  df_num["TrainAcc_std"])],
#         "Test Accuracy":   [fmt(m, s) for m, s in zip(df_num["TestAcc_mean"],   df_num["TestAcc_std"])],
#         "Balanced Acc.":   [fmt(m, s) for m, s in zip(df_num["BalAcc_mean"],    df_num["BalAcc_std"])],
#         "F1 Score":        [fmt(m, s) for m, s in zip(df_num["F1_mean"],        df_num["F1_std"])],
#         "Recall":          [fmt(m, s) for m, s in zip(df_num["Recall_mean"],    df_num["Recall_std"])],
#         "Precision":       [fmt(m, s) for m, s in zip(df_num["Precision_mean"], df_num["Precision_std"])],
#         "ROC-AUC":         [fmt(m, s) for m, s in zip(df_num["ROC_mean"],       df_num["ROC_std"])],
#         "PR-AUC":          [fmt(m, s) for m, s in zip(df_num["PRAUC_mean"],     df_num["PRAUC_std"])],
#         "Brier":           [fmt(m, s) for m, s in zip(df_num["Brier_mean"],     df_num["Brier_std"])],
#         "LogLoss":         [fmt(m, s) for m, s in zip(df_num["LogLoss_mean"],   df_num["LogLoss_std"])],
#     })
#     return df_view, df_num

# def display_scrollable(df_view, height=700):
#     """Zeigt df_view in einem scrollbaren Container (Jupyter-freundlich)."""
#     html = (
#         f'<div style="max-height:{height}px; overflow:auto; border:1px solid #ccc; '
#         f'padding:6px; background:transparent;">'
#         + df_view.to_html(index=False, border=0)
#         + '</div>'
#     )
#     display(HTML(html))


# # ─────────────────────────────── Main-Block ───────────────────────────────
# if __name__ == "__main__":
#     pd.set_option("display.width", 220)

#     file_path = r"K:\Team\Böhmer_Michael\PAPER\Philipp\Datensatz\Basistabelle_5_bis_14_PostOP_outlier.xlsx"
#     target_col = "Verletzungsstatus"

#     try:
#         df = pd.read_excel(file_path)

#         if target_col not in df.columns:
#             raise ValueError(f"Zielvariable '{target_col}' nicht in Datei gefunden.")

#         y = df[target_col]

#         # Objektspalten numerisch versuchen; anschließend nur numerische Features behalten
#         df_num = df.drop(columns=[target_col]).copy()
#         for c in df_num.columns:
#             if df_num[c].dtype == "object":
#                 df_num[c] = pd.to_numeric(df_num[c], errors="ignore")

#         X_df = df_num.select_dtypes(include=[np.number]).copy()
#         if X_df.shape[1] == 0:
#             raise ValueError("Keine numerischen Features gefunden.")

#         # Modelle definieren (wie gehabt)
#         models = {
#             "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
#             "Decision Tree": DecisionTreeClassifier(random_state=42),
#             "Random Forest": RandomForestClassifier(random_state=42),
#             "Gradient Boosting": GradientBoostingClassifier(random_state=42),
#             "XGBoost": XGBClassifier(eval_metric="logloss", random_state=42, use_label_encoder=False),
#             "LightGBM": LGBMClassifier(verbose=-1, random_state=42),
#             "SVC": SVC(probability=True, random_state=42),
#             "k-Nearest Neighbors": KNeighborsClassifier(),
#             "MLP Classifier": MLPClassifier(max_iter=1000, random_state=42),
#             "Gaussian Naive Bayes": GaussianNB(),
#             "Linear Discriminant Analysis": LinearDiscriminantAnalysis(),
#             "Quadratic Discriminant Analysis": QuadraticDiscriminantAnalysis(),
#             "Bagging Classifier": BaggingClassifier(random_state=42),
#             "Extra Trees": ExtraTreesClassifier(random_state=42),
#         }

#         # Cross-Val ausführen
#         results = {}
#         for name, mdl in models.items():
#             print(f"Validiere Modell: {name}")
#             metrics = repeated_k_fold_robust(mdl, X_df, y, n_splits=5, n_repeats=10)
#             results[name] = metrics

#         # Tabelle bauen & scrollbar darstellen
#         df_view, df_numeric = build_results_table(results)
#         display_scrollable(df_view, height=700)

#         # Optional: numerische Ergebnisse exportieren
#         # df_numeric.to_csv("model_cv_metrics_numeric.csv", index=False)

#     except FileNotFoundError:
#         print("Die Datei wurde nicht gefunden. Bitte überprüfe den Pfad.")
#     except Exception as e:
#         print(f"Ein Fehler ist aufgetreten: {e}")


# ============================ Robust CV + CI-Table ============================
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from IPython.display import HTML, display

from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, recall_score, precision_score,
    balanced_accuracy_score, average_precision_score, brier_score_loss, log_loss
)

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier, BaggingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis


# ---------------- Utilities ----------------
def proba_or_decision(model, X):
    """Bevorzuge predict_proba; sonst decision_function (auf [0,1] gemappt); sonst Pseudo-Score."""
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, "decision_function"):
        s = model.decision_function(X)
        s_min, s_max = np.min(s), np.max(s)
        return (s - s_min) / (s_max - s_min + 1e-12)
    return (model.predict(X) == 1).astype(float)


def empirical_ci(values, alpha=0.05):
    """Empirisches Perzentil-CI."""
    values = np.asarray(values, dtype=float)
    mean_val = np.mean(values)
    low = np.percentile(values, 100 * (alpha / 2))
    high = np.percentile(values, 100 * (1 - alpha / 2))
    return mean_val, low, high


def fmt_ci(mean_val, low, high, decimals=3):
    return f"{mean_val:.{decimals}f} ({low:.{decimals}f}–{high:.{decimals}f})"


def repeated_k_fold_robust(base_estimator, X_df: pd.DataFrame, y_vec, n_splits=5, n_repeats=10):
    # Binäres Label robust kodieren
    le = LabelEncoder()
    y = le.fit_transform(y_vec)
    if len(le.classes_) != 2:
        raise ValueError("Dieses Setup erwartet binäre Klassifikation (2 Klassen).")

    rkf = RepeatedStratifiedKFold(n_splits=n_splits, n_repeats=n_repeats, random_state=42)

    # Pipeline: Imputer + Scaler + Modell
    pipe = Pipeline(steps=[
        ("impute", SimpleImputer(strategy="median")),
        ("scale",  StandardScaler(with_mean=True, with_std=True)),
        ("clf",    base_estimator)
    ])

    # rohe Fold-Metriken sammeln
    m = {
        "acc_tr": [], "acc_te": [], "bacc": [],
        "f1": [], "rec": [], "prec": [],
        "roc": [], "prauc": [],
        "brier": [], "logloss": []
    }

    X = X_df.values

    for tr, te in rkf.split(X, y):
        Xtr, Xte = X[tr], X[te]
        ytr, yte = y[tr], y[te]

        pipe.fit(Xtr, ytr)

        ytr_pred = pipe.predict(Xtr)
        yte_pred = pipe.predict(Xte)
        yte_score = proba_or_decision(pipe, Xte)
        yte_score = np.clip(yte_score, 1e-7, 1 - 1e-7)  # stabil für Brier/LogLoss

        m["acc_tr"].append(accuracy_score(ytr, ytr_pred))
        m["acc_te"].append(accuracy_score(yte, yte_pred))
        m["bacc"].append(balanced_accuracy_score(yte, yte_pred))

        m["f1"].append(f1_score(yte, yte_pred, zero_division=0))
        m["rec"].append(recall_score(yte, yte_pred, zero_division=0))
        m["prec"].append(precision_score(yte, yte_pred, zero_division=0))

        m["roc"].append(roc_auc_score(yte, yte_score))
        m["prauc"].append(average_precision_score(yte, yte_score))

        m["brier"].append(brier_score_loss(yte, yte_score))
        m["logloss"].append(log_loss(yte, yte_score))

    return m


def build_results_table(results_dict, alpha=0.05):
    """
    Baut zwei DataFrames:
      - df_view: formatiert als mean (95%-CI)
      - df_num:  numerisch mit mean/low/high je Metrik
    Sortierung nach ROC-AUC mean absteigend.
    """
    rows_num = []

    for name, m in results_dict.items():
        acc_tr_mean, acc_tr_low, acc_tr_high = empirical_ci(m["acc_tr"], alpha=alpha)
        acc_te_mean, acc_te_low, acc_te_high = empirical_ci(m["acc_te"], alpha=alpha)
        bacc_mean, bacc_low, bacc_high = empirical_ci(m["bacc"], alpha=alpha)
        f1_mean, f1_low, f1_high = empirical_ci(m["f1"], alpha=alpha)
        rec_mean, rec_low, rec_high = empirical_ci(m["rec"], alpha=alpha)
        prec_mean, prec_low, prec_high = empirical_ci(m["prec"], alpha=alpha)
        roc_mean, roc_low, roc_high = empirical_ci(m["roc"], alpha=alpha)
        prauc_mean, prauc_low, prauc_high = empirical_ci(m["prauc"], alpha=alpha)
        brier_mean, brier_low, brier_high = empirical_ci(m["brier"], alpha=alpha)
        logloss_mean, logloss_low, logloss_high = empirical_ci(m["logloss"], alpha=alpha)

        rows_num.append({
            "Model": name,

            "TrainAcc_mean": acc_tr_mean, "TrainAcc_low": acc_tr_low, "TrainAcc_high": acc_tr_high,
            "TestAcc_mean": acc_te_mean, "TestAcc_low": acc_te_low, "TestAcc_high": acc_te_high,
            "BalAcc_mean": bacc_mean, "BalAcc_low": bacc_low, "BalAcc_high": bacc_high,
            "F1_mean": f1_mean, "F1_low": f1_low, "F1_high": f1_high,
            "Recall_mean": rec_mean, "Recall_low": rec_low, "Recall_high": rec_high,
            "Precision_mean": prec_mean, "Precision_low": prec_low, "Precision_high": prec_high,
            "ROC_mean": roc_mean, "ROC_low": roc_low, "ROC_high": roc_high,
            "PRAUC_mean": prauc_mean, "PRAUC_low": prauc_low, "PRAUC_high": prauc_high,
            "Brier_mean": brier_mean, "Brier_low": brier_low, "Brier_high": brier_high,
            "LogLoss_mean": logloss_mean, "LogLoss_low": logloss_low, "LogLoss_high": logloss_high,
        })

    df_num = pd.DataFrame(rows_num).sort_values("ROC_mean", ascending=False).reset_index(drop=True)

    df_view = pd.DataFrame({
        "Model": df_num["Model"],
        "Train Accuracy": [
            fmt_ci(m, lo, hi) for m, lo, hi in zip(df_num["TrainAcc_mean"], df_num["TrainAcc_low"], df_num["TrainAcc_high"])
        ],
        "Test Accuracy": [
            fmt_ci(m, lo, hi) for m, lo, hi in zip(df_num["TestAcc_mean"], df_num["TestAcc_low"], df_num["TestAcc_high"])
        ],
        "Balanced Acc.": [
            fmt_ci(m, lo, hi) for m, lo, hi in zip(df_num["BalAcc_mean"], df_num["BalAcc_low"], df_num["BalAcc_high"])
        ],
        "F1 Score": [
            fmt_ci(m, lo, hi) for m, lo, hi in zip(df_num["F1_mean"], df_num["F1_low"], df_num["F1_high"])
        ],
        "Recall": [
            fmt_ci(m, lo, hi) for m, lo, hi in zip(df_num["Recall_mean"], df_num["Recall_low"], df_num["Recall_high"])
        ],
        "Precision": [
            fmt_ci(m, lo, hi) for m, lo, hi in zip(df_num["Precision_mean"], df_num["Precision_low"], df_num["Precision_high"])
        ],
        "ROC-AUC": [
            fmt_ci(m, lo, hi) for m, lo, hi in zip(df_num["ROC_mean"], df_num["ROC_low"], df_num["ROC_high"])
        ],
        "PR-AUC": [
            fmt_ci(m, lo, hi) for m, lo, hi in zip(df_num["PRAUC_mean"], df_num["PRAUC_low"], df_num["PRAUC_high"])
        ],
        "Brier": [
            fmt_ci(m, lo, hi) for m, lo, hi in zip(df_num["Brier_mean"], df_num["Brier_low"], df_num["Brier_high"])
        ],
        "LogLoss": [
            fmt_ci(m, lo, hi) for m, lo, hi in zip(df_num["LogLoss_mean"], df_num["LogLoss_low"], df_num["LogLoss_high"])
        ],
    })

    return df_view, df_num


def display_scrollable(df_view, height=700):
    """Zeigt df_view in einem scrollbaren Container (Jupyter-freundlich)."""
    html = (
        f'<div style="max-height:{height}px; overflow:auto; border:1px solid #ccc; '
        f'padding:6px; background:transparent;">'
        + df_view.to_html(index=False, border=0)
        + '</div>'
    )
    display(HTML(html))


# ─────────────────────────────── Main-Block ───────────────────────────────
if __name__ == "__main__":
    pd.set_option("display.width", 220)

    file_path = r"K:\Team\Böhmer_Michael\PAPER\Philipp\Datensatz\Basistabelle_5_bis_14_PostOP_outlier.xlsx"
    target_col = "Verletzungsstatus"

    try:
        df = pd.read_excel(file_path)

        if target_col not in df.columns:
            raise ValueError(f"Zielvariable '{target_col}' nicht in Datei gefunden.")

        y = df[target_col]

        # Objektspalten numerisch versuchen; anschließend nur numerische Features behalten
        df_num = df.drop(columns=[target_col]).copy()
        for c in df_num.columns:
            if df_num[c].dtype == "object":
                df_num[c] = pd.to_numeric(df_num[c], errors="ignore")

        X_df = df_num.select_dtypes(include=[np.number]).copy()
        if X_df.shape[1] == 0:
            raise ValueError("Keine numerischen Features gefunden.")

        # Modelle definieren
        models = {
            "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
            "Decision Tree": DecisionTreeClassifier(random_state=42),
            "Random Forest": RandomForestClassifier(random_state=42),
            "Gradient Boosting": GradientBoostingClassifier(random_state=42),
            "XGBoost": XGBClassifier(eval_metric="logloss", random_state=42, use_label_encoder=False),
            "LightGBM": LGBMClassifier(verbose=-1, random_state=42),
            "SVC": SVC(probability=True, random_state=42),
            "k-Nearest Neighbors": KNeighborsClassifier(),
            "MLP Classifier": MLPClassifier(max_iter=1000, random_state=42),
            "Gaussian Naive Bayes": GaussianNB(),
            "Linear Discriminant Analysis": LinearDiscriminantAnalysis(),
            "Quadratic Discriminant Analysis": QuadraticDiscriminantAnalysis(),
            "Bagging Classifier": BaggingClassifier(random_state=42),
            "Extra Trees": ExtraTreesClassifier(random_state=42),
        }

        # Cross-Val ausführen
        results = {}
        for name, mdl in models.items():
            print(f"Validiere Modell: {name}")
            metrics = repeated_k_fold_robust(mdl, X_df, y, n_splits=5, n_repeats=10)
            results[name] = metrics

        # Tabelle bauen & scrollbar darstellen
        df_view, df_numeric = build_results_table(results, alpha=0.05)
        display_scrollable(df_view, height=700)

        # Optional: numerische Ergebnisse exportieren
        # df_numeric.to_csv("model_cv_metrics_numeric_ci.csv", index=False)

    except FileNotFoundError:
        print("Die Datei wurde nicht gefunden. Bitte überprüfe den Pfad.")
    except Exception as e:
        print(f"Ein Fehler ist aufgetreten: {e}")

Validiere Modell: Logistic Regression
Validiere Modell: Decision Tree
Validiere Modell: Random Forest
Validiere Modell: Gradient Boosting
Validiere Modell: XGBoost
Validiere Modell: LightGBM
Validiere Modell: SVC
Validiere Modell: k-Nearest Neighbors
Validiere Modell: MLP Classifier
Validiere Modell: Gaussian Naive Bayes
Validiere Modell: Linear Discriminant Analysis
Validiere Modell: Quadratic Discriminant Analysis
Validiere Modell: Bagging Classifier
Validiere Modell: Extra Trees


Model,Train Accuracy,Test Accuracy,Balanced Acc.,F1 Score,Recall,Precision,ROC-AUC,PR-AUC,Brier,LogLoss
Logistic Regression,1.000 (1.000–1.000),0.875 (0.765–0.987),0.872 (0.757–0.987),0.858 (0.714–0.987),0.815 (0.625–1.000),0.921 (0.722–1.000),0.971 (0.920–1.000),0.973 (0.932–1.000),0.081 (0.028–0.138),0.245 (0.111–0.398)
SVC,0.980 (0.970–1.000),0.886 (0.765–1.000),0.882 (0.752–1.000),0.867 (0.708–1.000),0.809 (0.625–1.000),0.946 (0.750–1.000),0.952 (0.836–1.000),0.959 (0.867–1.000),0.085 (0.021–0.164),0.283 (0.106–0.509)
XGBoost,1.000 (1.000–1.000),0.877 (0.719–0.941),0.873 (0.714–0.944),0.857 (0.677–0.941),0.806 (0.625–1.000),0.933 (0.756–1.000),0.943 (0.821–1.000),0.954 (0.874–1.000),0.093 (0.024–0.193),0.309 (0.109–0.623)
LightGBM,1.000 (1.000–1.000),0.857 (0.765–0.941),0.853 (0.752–0.944),0.838 (0.714–0.941),0.801 (0.625–1.000),0.897 (0.727–1.000),0.943 (0.847–1.000),0.951 (0.856–1.000),0.100 (0.044–0.171),0.317 (0.149–0.573)
Extra Trees,1.000 (1.000–1.000),0.887 (0.765–1.000),0.883 (0.750–1.000),0.867 (0.677–1.000),0.804 (0.528–1.000),0.956 (0.750–1.000),0.939 (0.838–1.000),0.950 (0.859–1.000),0.111 (0.065–0.164),0.377 (0.273–0.498)
MLP Classifier,1.000 (1.000–1.000),0.848 (0.692–0.987),0.843 (0.689–0.986),0.819 (0.627–0.987),0.758 (0.500–1.000),0.912 (0.714–1.000),0.935 (0.778–1.000),0.943 (0.837–1.000),0.114 (0.029–0.207),0.371 (0.102–0.808)
Random Forest,1.000 (1.000–1.000),0.885 (0.765–1.000),0.880 (0.752–1.000),0.863 (0.677–1.000),0.795 (0.528–1.000),0.961 (0.778–1.000),0.933 (0.838–1.000),0.945 (0.864–1.000),0.126 (0.088–0.164),0.417 (0.330–0.504)
Gaussian Naive Bayes,0.935 (0.896–0.970),0.857 (0.706–0.987),0.857 (0.710–0.987),0.847 (0.706–0.987),0.848 (0.653–1.000),0.861 (0.667–1.000),0.913 (0.753–1.000),0.897 (0.666–1.000),0.137 (0.012–0.294),1.428 (0.038–3.678)
Bagging Classifier,0.983 (0.970–1.000),0.833 (0.688–1.000),0.826 (0.688–1.000),0.790 (0.573–1.000),0.702 (0.403–1.000),0.937 (0.714–1.000),0.883 (0.735–1.000),0.896 (0.767–1.000),0.134 (0.058–0.231),0.623 (0.226–1.489)
Gradient Boosting,1.000 (1.000–1.000),0.779 (0.588–0.940),0.777 (0.590–0.938),0.759 (0.551–0.933),0.741 (0.500–0.972),0.796 (0.559–1.000),0.866 (0.689–0.997),0.883 (0.698–0.997),0.204 (0.063–0.399),1.422 (0.241–2.935)
